# Reproduction of Bahmani et al. (2012) — Tables 3/4, Figures 5.1/5.2

Driver: `src/paper_experiments.py`. Plan and decisions: `docs/ANALYSIS_PLAN.md`.

**Workflow**: this notebook runs on the **head VM** (jupyter via tunnel
`localhost:4444`). Before running: `git pull` of the branch
`repo-reorganization`. The sections marked **[CLUSTER]** require the cluster
started (`launch_cluster`); the **Fig 5.2** section also runs locally.

Default flag state: everything off (`RUN_FIG52 = False`, etc.) — turn on
one section at a time.

## 0. Sanity check del repository

In [ ]:
!git branch --show-current && git status --short && git log --oneline -3

## 1. Import e flag

In [ ]:
import os, time
import numpy as np
import pandas as pd

from src.launch_cluster import launch_cluster, shutdown_cluster
from src.data_loader import load_dataset, make_gauss_mixture, array_to_dask
from src.paper_experiments import (
    run_fig51, run_fig52, run_table34,
    plot_fig51, plot_fig52, table34_cost_table, table34_time_table,
)
from src.benchmark import RESULTS_DIR

# --- execution flags: turn on ONE section at a time ---
RUN_FIG52_TINY   = True   # reduced grid, local (~1 min)
RUN_FIG52_FULL   = False  # full paper grid, local (~hours)
RUN_CLUSTER      = True   # enables the [CLUSTER] sections
RUN_FIG51        = True   # KDD 10%, exact-l (cluster, moderate cost)
RUN_TABLE_SANITY = True   # single k=500 l/k=10 run BEFORE the full sweep
RUN_TABLE34      = True   # KDD full (cluster, OVERNIGHT)

SEED = 42
N_RUNS = 11                # paper convention: median over 11 runs

## 2. Fig 5.2 — GaussMixture **[LOCAL]**

Reduced validation grid before the full run.

In [ ]:
if RUN_FIG52_TINY:
    df52 = run_fig52(client=None,
                     R_values=(1,), l_over_k_values=(1.0, 2.0),
                     r_values=(0, 1, 2, 3), k=20, n=2_000, d=8,
                     n_runs=2, seed=SEED, max_iter_fit=30)
    df52.to_csv(os.path.join(RESULTS_DIR, "fig52_tiny.csv"), index=False)
    display(df52.groupby(["method", "l_over_k", "r"])["cost_final"].median())
else:
    print("RUN_FIG52_TINY = False")

Full run (paper protocol: R in {1,10,100}, l/k in {0.1,...,10},
r = 0..15, median over 11 runs, k=50). Warning: hours of local CPU.

In [ ]:
if RUN_FIG52_FULL:
    df52 = run_fig52(client=None, seed=SEED, n_runs=N_RUNS)
    _ts = time.strftime("%Y%m%d_%H%M%S")
    _csv = os.path.join(RESULTS_DIR, f"fig52_{_ts}.csv")
    df52.to_csv(_csv, index=False)
    print("Saved", _csv)
    plot_fig52(df52, output_dir="figures")
else:
    print("RUN_FIG52_FULL = False")

## 3. KDD data **[CLUSTER]** — loading (as in analysis.ipynb)

Paths identical to the VMs (DO NOT modify, see AGENTS.md). For the 10% use
`DATASET_URL_10PC`; for the tables use `DATASET_URL_FULL`.

In [ ]:
DATASET_URL_10PC = "https://ndownloader.figshare.com/files/5976042"
DATASET_URL_FULL = "https://ndownloader.figshare.com/files/5976045"

RAW_GZ_PATH  = "/home/ubuntu/Project/libero_development/data/kddcup_data.gz"
PARQUET_PATH = "/tmp/kddcup_data_shards"  # directory of Parquet shard files on master

COL_NAMES = [
    "duration","protocol_type","service","flag","src_bytes",
    "dst_bytes","land","wrong_fragment","urgent","hot",
    "num_failed_logins","logged_in","num_compromised","root_shell",
    "su_attempted","num_root","num_file_creations","num_shells",
    "num_access_files","num_outbound_cmds","is_host_login",
    "is_guest_login","count","srv_count","serror_rate",
    "srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate",
    "diff_srv_rate","srv_diff_host_rate","dst_host_count",
    "dst_host_srv_count","dst_host_same_srv_rate",
    "dst_host_diff_srv_rate","dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate","dst_host_serror_rate",
    "dst_host_srv_serror_rate","dst_host_rerror_rate",
    "dst_host_srv_rerror_rate","label",
]

### Cluster on/off

In [ ]:
# DO NOT RUN if already existing!
if RUN_CLUSTER:
    N_WORKERS = 8
    NUM_PARTITIONS = 8 * N_WORKERS
    cluster, client = launch_cluster(N_WORKERS)
else:
    print("RUN_CLUSTER = False")

## 4. Fig 5.1 — KDD 10%, exact-ℓ **[CLUSTER]**

Protocollo: k ∈ {17,33,65,129}, ℓ/k ∈ {1,2,4}, r = 1..10, mediana su 11 run.
Costo moderato: ~4·3·10·11 = 1320 run di seeding+Lloyd's sul 10%.

In [ ]:
if RUN_CLUSTER and RUN_FIG51:
    X_bag, _ = load_dataset(DATASET_URL_10PC, RAW_GZ_PATH, PARQUET_PATH,
                            col_names=COL_NAMES,
                            n_partitions=NUM_PARTITIONS, client=client)
    df51 = run_fig51(client, X_bag, seed=SEED, n_runs=N_RUNS,
                     num_partitions=NUM_PARTITIONS)
    df51.to_csv(os.path.join(RESULTS_DIR, "fig51_full.csv"), index=False)
    plot_fig51(df51, output_dir="figures")
else:
    print("RUN_CLUSTER/RUN_FIG51 = False")

## 5. Table 3/4 — KDD full **[CLUSTER, overnight]**

First the sanity check (risk #3 of ANALYSIS_PLAN: large candidate pools with
l=10k), then the full sweep. Critical protocol: `policy="fixed", r=5`
(handled inside `run_table34`) — the auto rule l/k<=0.1->15 is NOT the one
of the table.

In [ ]:
if RUN_CLUSTER and RUN_TABLE_SANITY:
    X_bag_full, _ = load_dataset(DATASET_URL_FULL, RAW_GZ_PATH, PARQUET_PATH,
                                 col_names=COL_NAMES,
                                 n_partitions=NUM_PARTITIONS, client=client)
    # singola run piu' pesante: k=500, l=5000 -> pool atteso ~25k candidati
    from src.paper_experiments import _run_one_parallel
    import time as _t
    _t0 = _t.time()
    res = _run_one_parallel(X_bag_full, k=500, l=5000, r=5, run_seed=SEED,
                            policy="fixed", max_iter_fit=10)
    print(res)
    print(f"totale {_t.time()-_t0:.1f}s")
else:
    print("RUN_CLUSTER/RUN_TABLE_SANITY = False")

In [ ]:
if RUN_CLUSTER and RUN_TABLE34:
    df34 = run_table34(client, X_bag_full, seed=SEED, n_runs=N_RUNS,
                       num_partitions=NUM_PARTITIONS)
    df34.to_csv(os.path.join(RESULTS_DIR, "table34_full.csv"), index=False)
else:
    print("RUN_CLUSTER/RUN_TABLE34 = False")

## 6. Analysis: paper-style figures and tables

From the saved CSVs (also works on this machine after bringing the CSVs
back into `results/`, or directly from the DataFrames of the sections above).

In [ ]:
# Example (uncomment when the CSVs exist):
# p51 = os.path.join(RESULTS_DIR, "fig51_full.csv")
# p52 = os.path.join(RESULTS_DIR, "fig52_<timestamp>.csv")
# p34 = os.path.join(RESULTS_DIR, "table34_full.csv")
# plot_fig51(p51, output_dir="figures")
# plot_fig52(p52, output_dir="figures")
# display(table34_cost_table(p34))     # Table 3, costs x1e-10 (medians)
# display(table34_time_table(p34))     # Table 4, times (medians)

## 7. Comparison with the paper values

Fill in after each artifact: obtained-vs-paper table (costs x10^-10,
medians). Reference values in the PDF: `docs/1203.6402v1.pdf`,
Tables 3/4 and Figures 5.1/5.2.

| Artefatto | Configurazione | Paper | Ottenuto | Note |
|---|---|---|---|---|
| Table 3 | k=500, ℓ/k=1, r=5, final | *(da PDF)* | | |
| ... | | | | |

## 8. Spegnimento cluster

In [ ]:
# Da eseguire a fine lavoro
shutdown_cluster(cluster, client)